## **import**

In [ ]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from groq import Groq
load_dotenv()

C:\Users\Admin\AppData\Local\Temp\ipykernel_18640\4243251098.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader
c:\Users\Admin\Desktop\RAG-learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# **1-ingestion pipeline**

## **load documents**

In [2]:
docs_data = "docs"
if not  os.path.exists(docs_data):
        raise FileNotFoundError("the data file was not found")

loader = DirectoryLoader(
        path = docs_data,
        glob = "*.txt",
        loader_cls = TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )

documents = loader.load()
if len(documents) == 0:
    raise FileNotFoundError(f"no .txt files in {docs_data}")
for i,doc in enumerate(documents[:2]):
    print(f"document {i+1}  :")
    print(doc.metadata['source'])

document 1  :
docs\Google.txt
document 2  :
docs\Microsoft.txt


## **chunk documents**

In [3]:
chunk_size = 800
chunk_overlap = 0
text_splitter = CharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap
    )
chunks = text_splitter.split_documents(documents)
for i,chunk in enumerate(chunks[:2]):
    print(f"--- chunk {i+1} ---")
    print(f"source : {chunk.metadata['source']}")
    print(f"length : {len(chunk.page_content)} characters")
    print(chunk.page_content)

Created a chunk of size 949, which is longer than the specified 800
Created a chunk of size 922, which is longer than the specified 800
Created a chunk of size 892, which is longer than the specified 800
Created a chunk of size 825, which is longer than the specified 800
Created a chunk of size 921, which is longer than the specified 800
Created a chunk of size 830, which is longer than the specified 800
Created a chunk of size 1055, which is longer than the specified 800
Created a chunk of size 874, which is longer than the specified 800
Created a chunk of size 1436, which is longer than the specified 800
Created a chunk of size 924, which is longer than the specified 800
Created a chunk of size 815, which is longer than the specified 800
Created a chunk of size 1039, which is longer than the specified 800
Created a chunk of size 1078, which is longer than the specified 800
Created a chunk of size 1043, which is longer than the specified 800
Created a chunk of size 880, which is longe

--- chunk 1 ---
source : docs\Google.txt
length : 600 characters
﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and technology
company focusing on online advertising, search engine
technology, cloud computing, computer software,
quantum computing, e-commerce, consumer
electronics, and artificial intelligence (AI).[9] It has
been referred to as "the most powerful company in the The Google logo used since 2015
world" by the BBC[10] and is one of the world's most
valuable brands.[11][12][13] Google's parent company,
Alphabet Inc., is one of the five Big Tech companies
alongside Amazon, Apple, Meta, and Microsoft.
--- chunk 2 ---
source : docs\Google.txt
length : 738 characters
Google was founded on September 4, 1998, by
American computer scientists Larry Page and Sergey
Brin. Together, they own about 14% of its publicly
listed shares and control 56% of its stockholder voting
power through super-voting stock. The company went
public via an in

## **create vector db + filling it with chunks**

In [4]:
dir = "db/chroma_db"
embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5"
    )
print("--- creating vector database ---")
vector_store = Chroma(
        persist_directory=dir,
        embedding_function=embedding_model,
        collection_metadata={"hnsw:space":"cosine"}
    )

vector_store.add_documents(chunks)
print("--- finished creating vector database ---")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5006.76it/s]


--- creating vector database ---
--- finished creating vector database ---


## **connecting to vector db**

In [7]:

dir = "db/chroma_db"

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vector_store = Chroma(
    persist_directory=dir,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4403.04it/s]


# **2-retrieval**

In [12]:

def retrieve(query):
    retriever = vector_store.as_retriever(search_kwargs={"k":3})

    relevant_docs = retriever.invoke(query)
    return relevant_docs

# **3-LLM**

In [14]:
groq_api_key = os.getenv("GROQ_API_KEY")
Client = Groq(api_key=groq_api_key)
history = []
while True:
    
    query = input("ask the question")
    if query == "quit":
            break
    if history:
        first_response = Client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role":"system",
                 "content":f"here is the historic of the questions  and answers {history} and here is the last question : {query}, if needed, reformulate the query so its contains enought context so we can answer it directly, return only the modified or not query and nothing else"}
            ],
            temperature=0
        )
        new_query = first_response.choices[0].message.content
    else:
         new_query = query
    print(f"new query : {new_query}")
    history.append(new_query)
    relevant_docs = retrieve(query=new_query)
    messages = [{
             "role":"system",
             "content":
             f"""
             here are some documents : 
            {chr(10).join([f"- {doc.page_content}" for doc in relevant_docs])}
            please provide an answer to the following question only based on the documents, if you dont have enought information in the 
            documents to answer, just say it
             """
           },
           {
               "role":"user",
               "content":f"question : {new_query}"
           }
    ]

    response = Client.chat.completions.create(
                    model="openai/gpt-oss-120b",
                    messages=messages,
                    temperature=0
                )
    content = response.choices[0].message.content
    history.append(content)
    print(content)


new query : what was microsoft first hardware release 
Microsoft’s first hardware release was the Microsoft Mouse, introduced in 1983.
new query : Did the Microsoft Mouse, Microsoft's first hardware release introduced in 1983, work well?
I don’t have enough information in the provided documents to determine how well the original Microsoft Mouse released in 1983 performed.
